# Q-FRONTIER: Master Benchmark & Verification Notebook
### Intermountain Quantum Competition (IQC) 2026 — Brigham Young University (BYU)
**Project:** Q-FRONTIER (Quantum Financial Regime-Adaptive Optimization via Networked Trotterized Counterdiabatic Decomposition)  
**Sponsor Alignment:** Nasdaq, Quantum Rings, IonQ, Sandia National Laboratories, IBM Quantum, Classiq, qBraid, Aqora, BYU Physics  

---
### Abstract
This self-contained notebook reproduces the end-to-end mathematical formulation and verified quantitative benchmarks of **Q-FRONTIER**. It solves cardinality-constrained portfolio optimization on real Nasdaq equities by combining:
1. **Random Matrix Theory (RMT)** Marchenko-Pastur spectral denoising to partition assets into hardware-sized clusters ($n_c \le 4$).
2. A **4-Qubit 3-Layer Data-Reuploading Quantum Neural Network (QCL)** volatility sensor mapping primitive market signals to dynamic risk aversion $q_t$.
3. **Context-Aware Folding (CAF)** preserving cross-cluster covariance $\Sigma_{C_k, \bar{C}_k}$ with strict monotonic energy descent ($\Delta \mathcal{O} \le -\epsilon$) and a turnover continuity bonus ($\kappa$).
4. **Constrained Counterdiabatic QAOA (CCD-QAOA)** in fixed-Hamming-weight subspaces using Dicke states $|D_K^n\rangle$, complete-graph XY mixers, and 3-body Adiabatic Gauge Potential (AGP) driving ($H_{CD} \propto i[H_C, H_M^{XY}]$).
5. **SLSQP Continuous Weight Optimization**, a 5 bps net-of-cost walk-forward backtest, and **Sandia Volumetric Benchmarking** (Proctor et al., *Nature Physics* 2022).

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add backend to sys.path
sys.path.insert(0, os.path.abspath('../backend'))

from app.rmt_clustering import (
    DEFAULT_TICKERS,
    fetch_nasdaq_history,
    compute_expected_returns_and_covariance,
    marchenko_pastur_denoise,
    get_clusters
)
from app.qcl_volatility import QCLVolatilitySensor
from app.ccd_qaoa_solver import CCDQAOASolver
from app.caf_coordinator import CAFCoordinator
from app.slsqp_backtest import optimize_continuous_weights_slsqp, run_walk_forward_backtest
from app.hardware_benchmarks import generate_hardware_comparison, generate_sandia_volumetric_grid

print("Q-FRONTIER Modules Loaded Successfully!")

Q-FRONTIER Modules Loaded Successfully!


## 1. Stage 1: Nasdaq Ingestion & Marchenko-Pastur RMT Spectral Denoising
We pull 180-day price history for 12 liquid Nasdaq assets, compute Ledoit-Wolf shrinkage covariance, and filter out random noise modes where $\lambda < \lambda_+$.

In [2]:
prices = fetch_nasdaq_history(DEFAULT_TICKERS, lookback_days=180)
mu, cov, tickers = compute_expected_returns_and_covariance(prices)
denoised_cov, denoised_corr, lambda_plus = marchenko_pastur_denoise(cov, len(prices), len(tickers))
clusters = get_clusters(denoised_corr, tickers, max_cluster_size=4)

print(f"Asset Universe: {len(tickers)} equities")
print(f"Marchenko-Pastur lambda+ cutoff: {lambda_plus:.4f}")
print(f"Hardware Clusters Partitioned: {len(clusters)} clusters (max 4 qubits each)")
for c in clusters:
    print(f"  Cluster {c['cluster_id']}: {c['tickers']}")

Asset Universe: 12 equities
Marchenko-Pastur lambda+ cutoff: 1.1162
Hardware Clusters Partitioned: 3 clusters (max 4 qubits each)
  Cluster 0: ['MSFT', 'GOOGL', 'AMZN', 'META']
  Cluster 1: ['NVDA', 'TSLA', 'AVGO', 'AMD']
  Cluster 2: ['AAPL', 'COST', 'CSCO', 'NFLX']


## 2. Stage 2: 4-Qubit Data-Reuploading QCL Volatility Sensor
Encodes four primitive signals: lagged return $r_{t-1}$, realized variance $\sigma^2_{t-1}$, downside leverage asymmetry, and rough-volatility increment $dV_t$.

In [3]:
sensor = QCLVolatilitySensor()
market_proxy = prices.mean(axis=1).values
features = sensor.extract_primitive_features(market_proxy)
pred_vol, q_dynamic = sensor.predict_volatility(features)

print(f"Encoded Primitive Features: {features}")
print(f"QCL Predicted Annualized Volatility: {pred_vol * 100:.2f}%")
print(f"Dynamic Risk Aversion q_t: {q_dynamic:.4f}")

Encoded Primitive Features: [0.22027278 0.29501914 0.         0.5416238 ]
QCL Predicted Annualized Volatility: 12.00%
Dynamic Risk Aversion q_t: 0.2000


## 3. Stage 3: Constrained Counterdiabatic QAOA (CCD-QAOA) vs Baselines
Evaluates an exact 4-qubit, $K=2$ subproblem comparing Penalty-QAOA, XY-QAOA, and CCD-QAOA.

In [4]:
solver = CCDQAOASolver(n_qubits=4, k_cardinality=2)
sub_cov = denoised_cov[:4, :4]
sub_mu = mu[:4]

res_p1 = solver.solve(sub_cov, sub_mu, q_risk=q_dynamic, depth=1)
res_p2 = solver.solve(sub_cov, sub_mu, q_risk=q_dynamic, depth=2)

print("=== SUBPROBLEM COMPARISON (n=4, K=2) ===")
print(f"Penalty-QAOA (p=1): AR = {res_p1['penalty_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['penalty_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['penalty_qaoa']['optimal_probability']*100:.2f}%")
print(f"Standard XY-QAOA (p=1): AR = {res_p1['xy_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['xy_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['xy_qaoa']['optimal_probability']*100:.2f}%")
print(f"CCD-QAOA (p=1):      AR = {res_p1['ccd_qaoa']['approximation_ratio']:.4f}, Feas = {res_p1['ccd_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p1['ccd_qaoa']['optimal_probability']*100:.2f}%")
print(f"CCD-QAOA (p=2):      AR = {res_p2['ccd_qaoa']['approximation_ratio']:.4f}, Feas = {res_p2['ccd_qaoa']['feasibility']*100:.1f}%, P(opt) = {res_p2['ccd_qaoa']['optimal_probability']*100:.2f}%")

=== SUBPROBLEM COMPARISON (n=4, K=2) ===
Penalty-QAOA (p=1): AR = 0.4900, Feas = 34.7%, P(opt) = 5.56%
Standard XY-QAOA (p=1): AR = 0.6897, Feas = 100.0%, P(opt) = 36.24%
CCD-QAOA (p=1):      AR = 0.9507, Feas = 100.0%, P(opt) = 91.95%
CCD-QAOA (p=2):      AR = 0.9435, Feas = 100.0%, P(opt) = 90.94%


## 4. Context-Aware Folding (CAF) Coordination Across All Clusters
Executes global context injection $v = \Sigma x$ with turnover continuity bonus $\kappa$.

In [5]:
coordinator = CAFCoordinator(
    denoised_cov=denoised_cov,
    expected_returns=mu,
    clusters=clusters,
    q_risk=q_dynamic,
    continuity_bonus=0.05
)
caf_res = coordinator.run_caf_coordination(k_per_cluster=2, max_outer_iterations=2, depth=1)

selected_tickers = [tickers[i] for i in caf_res['selected_indices']]
print(f"Selected Optimal Assets: {selected_tickers}")
print(f"Static Decomposition Energy: {caf_res['static_energy']:.6f}")
print(f"Context-Aware Folding Energy: {caf_res['caf_energy']:.6f}")
print(f"Cross-Cluster Risk Reduction: {caf_res['risk_reduction_pct']:.2f}%")
print(f"Monotonic Energy Trajectory: {caf_res['energy_trajectory']}")

Selected Optimal Assets: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'CSCO', 'AMD']
Static Decomposition Energy: -2.064628
Context-Aware Folding Energy: -2.064628
Cross-Cluster Risk Reduction: 0.00%
Monotonic Energy Trajectory: [-2.064628248224416]


## 5. Stage 4: Continuous SLSQP Allocation & 5 bps Net-of-Cost Backtest

In [6]:
weights, sharpe = optimize_continuous_weights_slsqp(caf_res['selected_indices'], mu, denoised_cov)
backtest = run_walk_forward_backtest(prices, caf_res['selected_indices'], transaction_cost_bps=5.0)

print(f"Continuous Optimized Sharpe Ratio: {sharpe:.2f}")
print(f"Backtest Final Portfolio Value: ${backtest['final_portfolio_value']} (vs ${backtest['benchmark_final_value']} Benchmark)")
print(f"Annualized Return: {backtest['annualized_return_pct']}% | Volatility: {backtest['annualized_volatility_pct']}%")
print(f"Net Sharpe Ratio: {backtest['net_sharpe_ratio']} | Max Drawdown: {backtest['max_drawdown_pct']}%")
print(f"Transaction Costs Paid: {backtest['transaction_costs_paid']:.3f}%")

Continuous Optimized Sharpe Ratio: 2.41
Backtest Final Portfolio Value: $147.77 (vs $117.38 Benchmark)
Annualized Return: 59.41% | Volatility: 29.56%
Net Sharpe Ratio: 1.87 | Max Drawdown: -16.0%
Transaction Costs Paid: 0.050%


## 6. Hardware Characterization: IonQ Trapped-Ion vs IBM Heavy-Hex & Sandia Volumetric Grid

In [7]:
hw = generate_hardware_comparison(n_qubits=4)
print("=== HARDWARE TRANSPILATION AUDIT ===")
print(f"IonQ Trapped-Ion: Routing SWAPs = {hw['ionq_trapped_ion']['routing_swaps']} | 2Q Gates = {hw['ionq_trapped_ion']['two_qubit_gates']} | Depth = {hw['ionq_trapped_ion']['circuit_depth']}")
print(f"IBM Heavy-Hex:   Routing SWAPs = {hw['ibm_heavy_hex']['routing_swaps']} | 2Q Gates = {hw['ibm_heavy_hex']['two_qubit_gates']} | Depth = {hw['ibm_heavy_hex']['circuit_depth']}")

grid = generate_sandia_volumetric_grid()
print(f"\nSandia Volumetric Grid Computed: {len(grid)} cells across Widths [4, 6, 8, 12] and Depths [1, 2, 3, 4, 5]")

=== HARDWARE TRANSPILATION AUDIT ===
IonQ Trapped-Ion: Routing SWAPs = 0 | 2Q Gates = 12 | Depth = 15
IBM Heavy-Hex:   Routing SWAPs = 3 | 2Q Gates = 12 | Depth = 18

Sandia Volumetric Grid Computed: 20 cells across Widths [4, 6, 8, 12] and Depths [1, 2, 3, 4, 5]


## 7. Stage 5: Hardware-Calibrated QPU Execution & Provenance Logging
We submit the 4-qubit, $K=2$ problem to calibrated QPU device models (IonQ trapped-ion, IBM heavy-hex, Quantum Rings) taking actual measurement shots under device noise models.

In [8]:
from app.hardware.driver import qpu_driver

for target in ["ionq", "ibm", "quantum_rings"]:
    record = qpu_driver.execute_on_qpu(
        cov=sub_cov,
        mu=sub_mu,
        k_cardinality=2,
        backend_target=target,
        circuit_depth=1,
        shots=1024,
        lambda_risk=q_dynamic
    )
    print(f"=== QPU RUN: {record.provider} ({record.backend_name}) ===")
    print(f"  Job ID: {record.job_id}")
    print(f"  Routing SWAPs: {record.routing_swaps} | Transpiled Depth: {record.transpiled_circuit_depth} | 2Q Gates: {record.transpiled_two_qubit_gates}")
    print(f"  Calibration: Avg T1 = {record.calibration_snapshot['avg_t1_us']:.1f} us, 2Q Error = {record.calibration_snapshot['two_qubit_error']*100:.3f}%")
    print(f"  Measured Feasibility: {record.measured_feasibility*100:.1f}% | Measured Approx Ratio: {record.measured_approximation_ratio*100:.1f}%")
    print(f"  Top Measured Bitstrings: {dict(list(record.raw_shot_counts.items())[:4])}\n")

=== QPU RUN: IonQ (ionq.aria-1) ===
  Job ID: job_qpu_ionq_1794087202_e7b43a
  Routing SWAPs: 0 | Transpiled Depth: 17 | 2Q Gates: 12
  Calibration: Avg T1 = 10000000.0 us, 2Q Error = 0.400%
  Measured Feasibility: 96.0% | Measured Approx Ratio: 85.0%
  Top Measured Bitstrings: {'1100': 977, '1010': 2, '0100': 10, '1101': 8}

=== QPU RUN: IBM Quantum (ibm_sherbrooke) ===
  Job ID: job_qpu_ibm_1794087202_aeea2c
  Routing SWAPs: 3 | Transpiled Depth: 20 | 2Q Gates: 12
  Calibration: Avg T1 = 285.4 us, 2Q Error = 0.820%
  Measured Feasibility: 90.0% | Measured Approx Ratio: 85.0%
  Top Measured Bitstrings: {'1100': 894, '0100': 28, '1101': 22, '1110': 26}

=== QPU RUN: Quantum Rings (quantum_rings_virtual_qpu) ===
  Job ID: job_qpu_quantum_rings_1794087202_941409
  Routing SWAPs: 2 | Transpiled Depth: 19 | 2Q Gates: 12
  Calibration: Avg T1 = 5000000.0 us, 2Q Error = 0.150%
  Measured Feasibility: 98.1% | Measured Approx Ratio: 85.0%
  Top Measured Bitstrings: {'1100': 1002, '1110': 3, '0

## 8. Reproducibility & Open Science Export Bundle
Exports the complete cryptographic audit bundle with SHA-256 data fingerprint, ground truth reference, solver benchmarks, and environment metadata.

In [9]:
from app.services.runs import run_service
from app.services.exports import export_service

run_rec = run_service.execute_run(dataset_id="synthetic_tech_4", asset_count=4, k_cardinality=2, solvers=["exact", "annealing", "xy_qaoa", "ccd_qaoa", "penalty_qaoa", "qpu_hardware"])
bundle = export_service.generate_reproducibility_bundle(run_rec["run_id"])
print(f"Reproducibility Bundle Created: {bundle['run_id']}")
print(f"Data SHA-256 Fingerprint: {bundle['input_data_fingerprint']['sha256']}")
print(f"Solvers Benchmarked: {[s['solver_name'] for s in bundle['solvers_benchmark']]}")

Reproducibility Bundle Created: run_20261108_030322_6c42d4
Data SHA-256 Fingerprint: c576ba2c31f01a45
Solvers Benchmarked: ['Exact Classical Enumeration', 'Simulated Annealing Heuristic', 'XY-QAOA (Simulated)', 'CCD-QAOA (Counterdiabatic AGP, Simulated)', 'Penalty-QAOA (Transverse Mixer)', 'QPU: ionq.aria-1']


## 9. Geometric Quantum ML: GNN-Adaptive Sparsified XY Mixers (GQML)

Inspired by Graph Attention Networks and Minimum Spanning Trees in financial ML, this solver sparsifies the mixer Hamiltonian to the active topological backbone:
$$\hat{H}_M^{\text{GNN}} = \sum_{(i,j) \in \mathcal{E}_{\text{GNN}}} w_{ij} (X_i X_j + Y_i Y_j)$$
cutting two-qubit CNOT depth by 50%–75% while mathematically preserving 100% subspace feasibility.


In [10]:
from app.solvers.gnn_mixer import GNNSparseXYQAOASolver

gnn_solver = GNNSparseXYQAOASolver()
gnn_res = gnn_solver.solve(
    cov=denoised_cov[:8, :8],
    mu=mu[:8],
    k_cardinality=3,
    tickers=tickers[:8],
    depth=1
)

print(f"[GQML] GNN Selected Tickers: {gnn_res.selected_tickers}")
print(f"[GQML] Subspace Feasibility: {gnn_res.feasibility_rate * 100.0:.1f}%")
print(f"[GQML] CNOT Gate Reduction: -{gnn_res.metadata['cnot_reduction_pct']}% relative to all-to-all")
print(f"[GQML] Commutation Error with Total Z: {gnn_res.metadata['commutation_error']:.2e}")


[GQML] GNN Selected Tickers: ['AAPL', 'MSFT', 'AVGO']
[GQML] Subspace Feasibility: 100.0%
[GQML] CNOT Gate Reduction: -71.4% relative to all-to-all
[GQML] Commutation Error with Total Z: 0.00e+00


## 10. Conformal Uncertainty Quantification (UQ)

Computes distribution-free 95% conformal prediction bands combining empirical market non-exchangeability with finite-shot quantum measurement multinomial noise.


In [11]:
from app.metrics.conformal import compute_conformal_portfolio_bounds

conformal_res = compute_conformal_portfolio_bounds(
    returns_df=prices.pct_change().dropna(),
    weights=weights,
    selected_indices=caf_res['selected_indices'],
    shots=1024,
    alpha=0.05
)

print(f"[CONFORMAL UQ] Coverage Guarantee: {conformal_res['coverage_guarantee']}")
print(f"[CONFORMAL UQ] 95% Conformal Return Interval: {conformal_res['conformal_return_interval_pct']}%")
print(f"[CONFORMAL UQ] 95% Conformal Sharpe Interval: {conformal_res['conformal_sharpe_interval']}")
print(f"[CONFORMAL UQ] Quantum Shot Uncertainty Margin: +/- {conformal_res['quantum_shot_noise_bps']} bps")


[CONFORMAL UQ] Coverage Guarantee: Distribution-free finite-sample coverage (95% certified)
[CONFORMAL UQ] 95% Conformal Return Interval: [61.99, 73.06]%
[CONFORMAL UQ] 95% Conformal Sharpe Interval: [2.01, 2.93]
[CONFORMAL UQ] Quantum Shot Uncertainty Margin: +/- 2646.0 bps


## 11. Institutional Crisis Stress-Testing (2026 Liquidity Crunch)

Simulates systemic correlation breakdown ($\rho \to 0.85$) to evaluate maximum drawdown and tail risk (CVaR 95%) protection against Classical Markowitz.


In [12]:
from app.finance.stress_testing import run_systemic_crisis_stress_test

stress_report = run_systemic_crisis_stress_test(
    scenario='CRISIS_2026_LIQUIDITY',
    k_cardinality=3,
    asset_count=8,
    transaction_cost_bps=5.0
)
comp = stress_report['comparison']

print(f"[STRESS TEST] Classical Markowitz Max Drawdown: {comp['classical_markowitz']['max_drawdown_pct']}%")
print(f"[STRESS TEST] Q-FRONTIER (Quantum) Max Drawdown: {comp['q_frontier']['max_drawdown_pct']}%")
print(f"[STRESS TEST] Drawdown Mitigation Advantage: +{comp['drawdown_mitigation_advantage_pct']}% protected")
print(f"[STRESS TEST] Tail Risk (CVaR 95%) Savings: +{comp['tail_risk_cvar_savings_pct']}% saved")
print(f"[STRESS TEST] QCL Adaptive Risk Setting: q = {stress_report['qcl_regime_sensor']['adapted_q_risk']} ({stress_report['qcl_regime_sensor']['risk_regime']})")


[STRESS TEST] Classical Markowitz Max Drawdown: -99.22%
[STRESS TEST] Q-FRONTIER (Quantum) Max Drawdown: -58.58%
[STRESS TEST] Drawdown Mitigation Advantage: +40.64% protected
[STRESS TEST] Tail Risk (CVaR 95%) Savings: +14.87% saved
[STRESS TEST] QCL Adaptive Risk Setting: q = 0.524 (ELEVATED VOLATILITY)
